# Val/Test Divergence Curves

Plots per-epoch AUPRC on the val set vs the original test set for each model.

Supports both benchmarks via the `BENCHMARK` switch below:
- **`anomaly`** (`anomaly_benchmark.py` → `divergence_curves.csv`): sources `original`, `synthetic-graph`, `real-subsampled-graph`, `original-cg`, `synthetic-cgt`.
- **`link`** (`link_benchmark.py` → `link_divergence_curves*.csv`): sources `original`, `synthetic-graph`, `original-cg`, `synthetic-cgt`. Sharded across the synthetic stem dir and the sibling `_original_cg_shared/` dir when run as an array job — both are auto-discovered and concatenated.

Both share the output layout `results/evaluate/{generator}/{dataset}/{task}/{stem}/`; only the CSV filenames differ.

**Interpreting the curves:**
- **Val:** AUPRC on the val set the model trained against — what early stopping optimises.
- **Test (original):** AUPRC on original-graph test items — the actual utility signal.
- The **divergence point** is where val keeps climbing while test plateaus or drops.
  A large gap indicates the synthetic / CG construction does not faithfully capture the original's structure.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from pathlib import Path

# --- Configure which run to analyse ---
GENERATOR      = 'cgt'
DATASET        = 'tolokers'
SYNTHETIC_NAME = 'tolokers_e50_k512_c1_d2_f5_s8818'
# Which benchmark produced the results — selects which CSV filenames to load.
# 'anomaly' → evaluation_results.csv + divergence_curves.csv
# 'link'    → link_prediction_results*.csv + link_divergence_curves*.csv
#             (the link array job shards by (phase_tag, eval_mode); shards
#              are auto-discovered across the synthetic stem dir and the
#              sibling `_original_cg_shared/` dir and concatenated.)
BENCHMARK      = 'anomaly'
# The --task value used by the benchmark run. Both benchmarks share the
# layout results/evaluate/{generator}/{dataset}/{task}/{stem}/. Defaults
# are 'hidden_labels' (anomaly) and 'hidden_links' (link). Leave empty to
# load legacy results that predate the per-task subdir.
TASK           = 'hidden_labels' if BENCHMARK == 'anomaly' else 'hidden_links'

# Resolve project root — works whether Jupyter is launched from root or scripts/benchmark/
_cwd = Path.cwd()
PROJ_ROOT   = _cwd if (_cwd / 'scripts').exists() else (_cwd / '../..').resolve()
_base_parts = ['results', 'evaluate', GENERATOR, DATASET]
if TASK:
    _base_parts.append(TASK)
_base_parts.append(SYNTHETIC_NAME)
RESULTS_DIR = PROJ_ROOT.joinpath(*_base_parts)
OUT_DIR     = RESULTS_DIR / 'divergence_plots'
OUT_DIR.mkdir(parents=True, exist_ok=True)


def _concat_csvs(paths, key_cols):
    """Read each CSV and concat; drop duplicate rows on key_cols (later shards win)."""
    if not paths:
        return None
    frames = [pd.read_csv(p) for p in paths]
    out = pd.concat(frames, ignore_index=True)
    return out.drop_duplicates(subset=key_cols, keep='last').reset_index(drop=True)


if BENCHMARK == 'link':
    # The link array job shards files by (phase_tag, eval_mode). Globs match
    # the legacy unsharded file and the array shards
    # (`__phase2only_synthetic_cgt.csv`, `__allphases_original_cg.csv`).
    # The original_cg shard lives in the sibling `_original_cg_shared/` dir.
    shared_dir = RESULTS_DIR.parent / '_original_cg_shared'
    curves_paths = sorted(RESULTS_DIR.glob('link_divergence_curves*.csv'))
    eval_paths   = sorted(RESULTS_DIR.glob('link_prediction_results*.csv'))
    if shared_dir.exists():
        curves_paths += sorted(shared_dir.glob('link_divergence_curves*.csv'))
        eval_paths   += sorted(shared_dir.glob('link_prediction_results*.csv'))
    df      = _concat_csvs(curves_paths, ['source', 'dataset', 'model', 'epoch'])
    eval_df = _concat_csvs(eval_paths,   ['source', 'dataset', 'model'])
    if df is None or eval_df is None:
        raise FileNotFoundError(
            f"No link_*.csv shards found under:\n  {RESULTS_DIR}\n"
            f"  {shared_dir}  (sibling, only checked if it exists)\n"
            f"Run the link benchmark first.")
    print('Curves shards:')
    for p in curves_paths: print(f'  {p.relative_to(PROJ_ROOT)}')
    print('Eval shards:')
    for p in eval_paths:   print(f'  {p.relative_to(PROJ_ROOT)}')
else:
    CURVES_PATH = RESULTS_DIR / 'divergence_curves.csv'
    EVAL_PATH   = RESULTS_DIR / 'evaluation_results.csv'
    df      = pd.read_csv(CURVES_PATH)
    eval_df = pd.read_csv(EVAL_PATH)

print(f'Curves: {len(df)} rows')
print(f'  sources:  {df.source.unique().tolist()}')
print(f'  models:   {df.model.unique().tolist()}')
print(f'Eval:   {len(eval_df)} rows, models: {eval_df.model.unique().tolist()}')

# Models that have per-epoch curves vs models plotted as final-AUPRC reference lines
CURVE_MODELS   = sorted(df.model.unique().tolist())
ALL_MODELS     = sorted(eval_df.model.unique().tolist())
NOCURVE_MODELS = [m for m in ALL_MODELS if m not in CURVE_MODELS]
print(f'Curve models:               {CURVE_MODELS}')
print(f'No-curve (final-AUPRC ref): {NOCURVE_MODELS}')
df.head()

In [ ]:
SOURCE_COLORS = {
    'original':              'steelblue',
    'synthetic-graph':       'tomato',
    'real-subsampled-graph': 'forestgreen',
    'original-cg':           'mediumorchid',
    'synthetic-cgt':         'darkorange',
}
SOURCE_LABELS = {
    'original':              'original',
    'synthetic-graph':       'synthetic',
    'real-subsampled-graph': 'real-subsampled',
    'original-cg':           'original-CG',
    'synthetic-cgt':         'synthetic-CGT',
}


def _plot_curve_model(ax, grp):
    """Plot val (solid) + test (dashed) curves for each source on a single axis."""
    for src, color in SOURCE_COLORS.items():
        sgrp = grp[grp['source'] == src]
        if sgrp.empty:
            continue
        label = SOURCE_LABELS[src]
        ax.plot(sgrp['epoch'], sgrp['val_auprc_mean'],
                label=f'Val — {label}', color=color, linestyle='-')
        ax.fill_between(sgrp['epoch'],
                        sgrp['val_auprc_mean'] - sgrp['val_auprc_std'],
                        sgrp['val_auprc_mean'] + sgrp['val_auprc_std'],
                        alpha=0.15, color=color)
        ax.plot(sgrp['epoch'], sgrp['test_auprc_mean'],
                label=f'Test — {label}', color=color, linestyle='--')
        ax.fill_between(sgrp['epoch'],
                        sgrp['test_auprc_mean'] - sgrp['test_auprc_std'],
                        sgrp['test_auprc_mean'] + sgrp['test_auprc_std'],
                        alpha=0.15, color=color)


def _plot_nocurve_model(ax, dataset, model, x_max):
    """XGB models: only final test AUPRC is saved. Plot horizontal test reference lines."""
    rows = eval_df[(eval_df['dataset'] == dataset) & (eval_df['model'] == model)]
    for src, color in SOURCE_COLORS.items():
        r = rows[rows['source'] == src]
        if r.empty:
            continue
        m = float(r['AUPRC_mean'].iloc[0])
        s = float(r['AUPRC_std'].iloc[0])
        label = SOURCE_LABELS[src]
        ax.hlines(m, 0, x_max, color=color, linestyle='--',
                  label=f'Test — {label} (final)')
        ax.fill_between([0, x_max], m - s, m + s, alpha=0.12, color=color)


# Use the curve-models' epoch range as the x-axis span for horizontal reference plots
x_max = int(df['epoch'].max()) if not df.empty else 100

# 1. Models with per-epoch curves (GCN/GIN/GraphSAGE)
for (dataset, model), grp in df.groupby(['dataset', 'model']):
    fig, ax = plt.subplots(figsize=(7, 4))
    _plot_curve_model(ax, grp)
    ax.set_xlabel('Epoch')
    ax.set_ylabel('AUPRC')
    ax.set_title(f'{dataset} — {model}')
    ax.legend(fontsize=8, ncol=2)
    ax.xaxis.set_major_locator(mticker.MaxNLocator(integer=True))
    plt.tight_layout()

    out_path = OUT_DIR / f'{dataset}_{model}.png'
    plt.savefig(out_path, dpi=150)
    plt.show()
    plt.close(fig)
    print(f'Saved {out_path}')

# 2. Models without curves (XGBGraph/XGBoost) — final test AUPRC as horizontal reference
for model in NOCURVE_MODELS:
    for dataset in eval_df['dataset'].unique():
        if eval_df[(eval_df['dataset'] == dataset) & (eval_df['model'] == model)].empty:
            continue
        fig, ax = plt.subplots(figsize=(7, 4))
        _plot_nocurve_model(ax, dataset, model, x_max)
        ax.set_xlabel('Epoch (n/a — single-fit model)')
        ax.set_ylabel('AUPRC')
        ax.set_title(f'{dataset} — {model} (final test AUPRC)')
        ax.legend(fontsize=8)
        ax.xaxis.set_major_locator(mticker.MaxNLocator(integer=True))
        plt.tight_layout()

        out_path = OUT_DIR / f'{dataset}_{model}.png'
        plt.savefig(out_path, dpi=150)
        plt.show()
        plt.close(fig)
        print(f'Saved {out_path}')

In [ ]:
# Divergence summary: at the epoch early stopping picks (peak val), what is test?
# val_test_gap > 0 means val overestimates true test performance at that checkpoint.
# XGB rows have NaN val/gap — only the final test AUPRC is recorded for single-fit models.
rows = []

# Curve models: best val epoch + val/test at that epoch
for (dataset, model, source), sgrp in df.groupby(['dataset', 'model', 'source']):
    best_row     = sgrp.iloc[sgrp['val_auprc_mean'].values.argmax()]
    val_at_best  = best_row['val_auprc_mean']
    test_at_best = best_row['test_auprc_mean']
    rows.append({
        'dataset': dataset, 'model': model, 'source': source,
        'best_val_epoch': int(best_row['epoch']),
        'val_auprc':    val_at_best,
        'test_auprc':   test_at_best,
        'val_test_gap': val_at_best - test_at_best,
    })

# No-curve models: only final test AUPRC available; val and gap not tracked
for _, r in eval_df.iterrows():
    if r['model'] in CURVE_MODELS:
        continue
    rows.append({
        'dataset': r['dataset'], 'model': r['model'], 'source': r['source'],
        'best_val_epoch': pd.NA,
        'val_auprc':      pd.NA,
        'test_auprc':     r['AUPRC_mean'],
        'val_test_gap':   pd.NA,
    })

summary = pd.DataFrame(rows).sort_values(['dataset', 'model', 'source'])
print('Summary — val_test_gap: how much val overestimates true test performance at early-stopping epoch')
print('NaN entries are XGB models — only final test AUPRC was recorded; val curve not tracked.')
summary.round(4)